#### Tool Calling

##### Tool Binding

**Tool Binding** is the step where you **register tools with a Language Model (LLM)** so that:

1. The LLM Know what **tools are avaiable**
2. It knows **what each tool does** (via description)
3. It knows **what input format to use** (via schema)

In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

tool create

In [2]:
@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [3]:
print(multiply.invoke({'a':3, 'b':4}))

multiply.name
multiply.description
multiply.args

12


{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

tool binding

In [4]:
# llm = ChatGroq(model="openai/gpt-oss-120b")

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [5]:
llm.invoke('hi')

AIMessage(content=[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'El4KXAERTTIPPSbYeo/3oBcLxZ2U2jjLCg1LX7mEG//4fPEA7H9/WDuAHkl0LjuEo8rKaph6Nr8MCWDdQk1kdV9FmK+mMihFJHweNKlkkGfvqNaPnx0w+XR/91nm00Bv'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0666f-adea-77e1-948f-aaa91fbfd253-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 9, 'total_tokens': 11, 'input_token_details': {'cache_read': 0}})

In [6]:
llm_with_tools = llm.bind_tools([multiply])

result = llm_with_tools.invoke("What is 3 multiplied by 4?")

result

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"b": 4, "a": 3}'}, '__gemini_function_call_thought_signatures__': {'call_1275689': 'El4KXAERTTIPokboZXFMNgcm9M2wydq6jE3nHtcDoK8dbkMDVf1lr7TJGRbS8uc+i9yFa+PogUQ8q/KMSIjPUOYHp5pw/v5y9SJ2LgwonhmHdzXxqm1EkKlMWB+mA4Ty'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0666f-bcba-70b0-9bb8-023b702c9952-0', tool_calls=[{'name': 'multiply', 'args': {'b': 4, 'a': 3}, 'id': 'call_1275689', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 69, 'output_tokens': 16, 'total_tokens': 85, 'input_token_details': {'cache_read': 0}})

### Tool Calling

**Tool Calling** is a process where the LLM decide, during a conversation or task, that it needs to use a specific tool(function) -- and generate a structured output with:

- the **name of the tool**
- and the **arguments** to call it with

The LLm **does not actually run the tool** -- it just **suggests** the tool and the input arguments. the **actual execution** is **handled by LangChain or you**

Ex: 
    "What's 8 multiple by 7?"

The LLM responds with a tool call:
```
{
    "tool": "multiply",
    "args": {"a":8, "b":7}
}
```

In [17]:
query = HumanMessage("What is 3 multiplied by 4?")

In [18]:
messages = [query]

In [19]:
messages

[HumanMessage(content='What is 3 multiplied by 4?', additional_kwargs={}, response_metadata={})]

In [ ]:
llm_with_tools.invoke("What is 3 multiplied by 4? Please use the multiply tool to calculate the answer.")

#### Tool Execution

**Tool Execution** is the step where the **actual python function (tool)** is run using the input argument that the **LLM suggested during tool calling**

In simpler words:

**The LLM Says:**
 - *Hey, call the multiply tool with a=8 and b=7*

**Tool Execution** is when you or langchain actually run:
 - multiply(a=8, b=7)

-> and get the result : 56

In [13]:
result = llm_with_tools.invoke("What is 3 multiplied by 4? Please use the multiply tool to calculate the answer.")


result.tool_calls[0]['args']
result.tool_calls[0]

{'name': 'multiply',
 'args': {'b': 4, 'a': 3},
 'id': 'call_1296133',
 'type': 'tool_call'}

In [14]:
# you can also invoke the tool directly with the arguments returned from the tool call and get the result
multiply.invoke(result.tool_calls[0]['args'])

12

In [15]:
# we can also pass the tool call to the llm_with_tools.invoke method and get the result

multiply.invoke({'name': 'multiply',
 'args': {'b': 4, 'a': 3},
 'id': 'call_1296133',
 'type': 'tool_call'})  

ToolMessage(content='12', name='multiply', tool_call_id='call_1296133')

In [ ]:
# in above result we get the ToolMessage, and this get's when we execute the tool with the help of tool call.
# and we can pass this tool message to the llm
